# Lost-words / Zgubione słowa — Colab local video pipeline

Polski pipeline do pionowych filmów 60–120 sekund (cel co najmniej 60 sekund, domyślnie około 75 sekund): polski scenariusz i TTS, gęsty montaż ujęć 1–3 s, lokalny renderer na GPU T4 lub CPU, checkpointy Google Drive i eksport 1080×1920.

Przepływ: scenariusz JSON po polsku → osobny polski głos dla każdej kwestii → manifest wielu krótkich ujęć → lokalny renderer Claude → osobne pliki MP4 i checkpointy na Google Drive → FFmpeg → finalny MP4 1080×1920.

Wypowiedzi, narracja, tytuły i napisy są po polsku. Prompty wizualne i ruchu są po angielsku. Sceny mają typ talking_character (widoczna postać wykonująca konkretną czynność) lub environment (ruch otoczenia). Twarz mówiącej postaci pozostaje widoczna w ujęciach medium_action/reaction; ewentualny lokalny lip-sync jest opcjonalny.

## Długość i rytm montażu

- Zakres akceptowany: 60–120 sekund.
- Cel: minimum 60 sekund; domyślnie około 75 sekund.
- Każde ujęcie trwa 1–3 sekundy. Każda kwestia jest dzielona na krótkie klipy o różnych planach.
- Każde ujęcie jest osobnym plikiem wideo zapisanym na Google Drive. Nie wydłużamy jednego statycznego klipu i nie symulujemy cięć zmianą FPS.
- Finalny film ma 1080×1920, 30 fps, H.264/AAC i napisy z DejaVu Sans.

## Tryb testowy

Domyślnie TEST_MODE = True. Ten tryb montuje Drive, tworzy polski scenariusz, generuje Edge TTS, wycina fragmenty audio 1–3 s i zapisuje manifest. Pomija lokalny model wizualny i nie tworzy MP4.

## Pełny lokalny render

Aby wygenerować wideo, ustaw TEST_MODE = False w komórce 2. Notebook ładuje visuals/shot_renderer.py, przekazuje mu pojedyncze ujęcia i profil sprzętowy, a następnie zapisuje każdy gotowy klip i checkpoint na Drive. Do renderowania nie jest wymagany żaden klucz API; komórki nie wysyłają płatnych zadań inference.

Preferowany jest CUDA/GPU T4. Notebook sprawdza wolną VRAM i dobiera pionową rozdzielczość pośrednią; w przypadku braku CUDA wybiera CPU. TPU v5e-1 nie jest używany w tym adapterze. Finalne skalowanie do 1080×1920 wykonuje FFmpeg.

Wagi modelu mogą zostać pobrane przy pierwszym uruchomieniu, jeżeli ich licencja zezwala na lokalne użycie; cache trafia do models/ na Drive. Opcjonalny OPENAI_API_KEY dotyczy wyłącznie dynamicznego scenariusza, może generować opłaty u dostawcy i nie jest potrzebny do domyślnego scenariusza demonstracyjnego.

Colab nie gwarantuje stałej dostępności GPU ani długości sesji. Po przerwaniu użyj tego samego EPISODE_ID_OVERRIDE i RESUME_RENDERING = True; poprawne ujęcia zostaną pominięte.

## Kontrakt renderera i podział pracy

Notebook utrzymuje komórki 1–5 i 7 oraz stabilny adapter w komórce 6. Claude rozwija wyłącznie visuals/shot_renderer.py i ewentualne pliki pomocnicze w visuals/. Nie należy równocześnie edytować tych samych komórek notebooka.

Interfejs:
\`\`\`python
render_shot(shot: dict, runtime: dict) -> dict
\`\`\`

Wejście zawiera shot_id, scene_id, duration_sec (1–3), visual_prompt_en, motion_prompt_en, audio_clip_path, output_path, frame_path, checkpoint_path i wymiary renderowania dobrane do VRAM. Wyjście zawiera status="completed", video_path, duration_sec, width, height, fps, backend, device i opcjonalne lipsync_applied.

Szczegóły kontraktu, struktura Drive i zasady merge są w docs/visual_renderer_contract.md.

## Sekrety

Domyślnie żadne sekrety nie są wymagane. OPENAI_API_KEY jest opcjonalny tylko dla zewnętrznego generatora scenariusza; Edge TTS nie wymaga klucza. Nie wpisuj sekretów do notebooka ani repozytorium.


In [ ]:
# KOMÓRKA 1/7 — Google Drive, FFmpeg, biblioteki i wspólne funkcje API
from pathlib import Path
import os, sys, json, re, math, time, random, shutil, subprocess, textwrap, importlib.util
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    raise RuntimeError(
        "Nie udało się zamontować Google Drive. Uruchom notebook w Colab i zaakceptuj dostęp do Dysku. "
        + str(exc)
    ) from exc

PROJECT_ROOT = Path('/content/drive/MyDrive/lost_words_project')
ASSETS_DIR = PROJECT_ROOT / 'assets'
AUDIO_DIR = PROJECT_ROOT / 'audio'
FRAMES_DIR = PROJECT_ROOT / 'frames'
VIDEO_DIR = PROJECT_ROOT / 'video'
TEMP_DIR = PROJECT_ROOT / 'temp'
OUTPUT_DIR = PROJECT_ROOT / 'output'
CHECKPOINTS_ROOT = PROJECT_ROOT / 'checkpoints'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
MODELS_DIR = PROJECT_ROOT / 'models'
for folder in (PROJECT_ROOT, ASSETS_DIR, AUDIO_DIR, FRAMES_DIR, VIDEO_DIR, TEMP_DIR, OUTPUT_DIR, CHECKPOINTS_ROOT, VISUALS_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def run_cmd(args, *, check=True, capture_output=True):
    """Uruchamia proces z diagnostyką stderr."""
    try:
        return subprocess.run(
            [str(x) for x in args],
            check=check,
            text=True,
            stdout=subprocess.PIPE if capture_output else None,
            stderr=subprocess.PIPE if capture_output else None,
        )
    except subprocess.CalledProcessError as exc:
        print(f'[BŁĄD] Proces {args[0]} zakończył się kodem {exc.returncode}.')
        if exc.stderr:
            print(exc.stderr[-5000:])
        raise
    except FileNotFoundError as exc:
        raise RuntimeError(f'Nie znaleziono programu {args[0]}: {exc}') from exc


try:
    if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
        run_cmd(['apt-get', 'update', '-qq'], capture_output=False)
        run_cmd([
            'apt-get', 'install', '-y', '-qq',
            'ffmpeg', 'fonts-dejavu-core', 'fontconfig'
        ], capture_output=False)
    FONT_FILE = Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
    if not FONT_FILE.exists():
        run_cmd(['apt-get', 'install', '-y', '-qq', 'fonts-dejavu-core'], capture_output=False)
    if not FONT_FILE.exists():
        raise RuntimeError(f'Brak czcionki DejaVu Sans: {FONT_FILE}')
except Exception as exc:
    raise RuntimeError(f'Instalacja FFmpeg/fontów nie powiodła się: {exc}') from exc

try:
    run_cmd([
        sys.executable, '-m', 'pip', 'install', '-q',
        'requests>=2.31.0', 'Pillow>=10.0.0', 'edge-tts>=6.1.0'
    ])
    import requests
    from PIL import Image
except Exception as exc:
    raise RuntimeError(f'Instalacja/import pakietów nie powiodły się: {exc}') from exc

print('[OK] FFmpeg:', run_cmd(['ffmpeg', '-version']).stdout.splitlines()[0])
print('[OK] DejaVu Sans:', FONT_FILE)
print('[OK] Projekt:', PROJECT_ROOT)


def probe_duration(path):
    result = run_cmd([
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(path),
    ])
    duration = float(result.stdout.strip())
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError(f'Niepoprawna długość pliku {path}: {duration}')
    return duration



def write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + '.tmp')
    temp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temp_path, path)
    return path


def read_json_if_exists(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError) as exc:
        print(f'[OSTRZEŻENIE] Nie można odczytać checkpointu {path.name}: {exc}')
        return default


def probe_media(path):
    result = run_cmd([
        'ffprobe', '-v', 'error', '-show_streams', '-show_format',
        '-of', 'json', str(path)
    ])
    data = json.loads(result.stdout)
    video = next((x for x in data.get('streams', []) if x.get('codec_type') == 'video'), None)
    audio = next((x for x in data.get('streams', []) if x.get('codec_type') == 'audio'), None)
    return {'data': data, 'video': video, 'audio': audio}


In [ ]:
# KOMÓRKA 2/7 — konfiguracja, TEST_MODE i sekrety

def get_secret(name, default=None):
    """Pobieranie Colab Secrets z awaryjnym odczytem z os.environ."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = os.environ.get(name)
    return value if value else default


SERIES_NAME = 'Lost-words / Zgubione słowa'
EPISODE_TOPIC = 'Masz osiem godzin, by dotrzeć do punktu ewakuacji. Na stacji pojawia się peron, którego nie ma w żadnym oficjalnym wykazie.'
EPISODE_LOCATION = 'Nieczynna stacja kolejowa i puste przedmieścia Łodzi'
NUM_SCENES = 12
MIN_VIDEO_SECONDS = 60
GOAL_VIDEO_SECONDS = 60
MAX_VIDEO_SECONDS = 120
TARGET_SECONDS = 75
MIN_SCRIPT_WORDS = 180
VIDEO_WIDTH, VIDEO_HEIGHT, FPS = 1080, 1920, 30
STORY_STYLE_PROFILE = (
    'Immersive what-if survival scenario: a clear premise, a ticking deadline, limited resources, '
    'cause-and-effect escalation, one unsettling contradiction, and a consequential final choice or twist. '
    'Address the viewer in second person when natural. Keep the scenario grounded and cinematic.'
)
VISUAL_STYLE = (
    'cinematic grounded survival thriller, near-future speculative realism, documentary camera language, '
    'desaturated natural colors, subtle 35mm film grain, practical lighting, atmospheric locations, '
    'realistic human performances, expressive close-ups and purposeful environmental shots'
)

# TEST_MODE=True przygotowuje scenariusz/TTS/manifest, ale pomija model wizualny.
# TEST_MODE=False uruchamia lokalny renderer Claude; notebook nie wysyła płatnych zadań API.
TEST_MODE = True
DRY_RUN = TEST_MODE
RESUME_RENDERING = True
LLM_PROVIDER = 'fallback'            # można zmienić na openai_compatible
TTS_PROVIDER = 'edge'
PAUSE_BETWEEN_LINES = 0.0
AMBIENT_MIX_GAIN = 0.12
RADIO_STATIC_GAIN = 0.08
SHOT_DURATION_MIN = 1.0
SHOT_DURATION_TARGET = 2.0
SHOT_DURATION_MAX = 3.0
VISUAL_RENDERER_PATH = VISUALS_DIR / 'shot_renderer.py'
VISUAL_RENDERER_SOURCE_URL = 'https://raw.githubusercontent.com/simonexex544/lost-words-video-pipeline/main/visuals/shot_renderer.py'
EPISODE_ID_OVERRIDE = ''  # wpisz poprzedni EPISODE_ID, aby wznowić render

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
OPENAI_BASE_URL = (get_secret('OPENAI_BASE_URL', 'https://api.openai.com/v1') or '').rstrip('/')
OPENAI_CHAT_MODEL = get_secret('OPENAI_CHAT_MODEL', 'gpt-4o-mini')
if not 10 <= NUM_SCENES <= 18:
    raise ValueError('NUM_SCENES ustaw w zakresie 10–18.')
if not MIN_VIDEO_SECONDS <= TARGET_SECONDS <= MAX_VIDEO_SECONDS:
    raise ValueError('TARGET_SECONDS musi mieścić się w zakresie 40–120 sekund.')
if not 0.05 <= AMBIENT_MIX_GAIN <= 0.30:
    raise ValueError('AMBIENT_MIX_GAIN ustaw w zakresie 0.05–0.30.')
if not SHOT_DURATION_MIN <= SHOT_DURATION_TARGET <= SHOT_DURATION_MAX:
    raise ValueError('Docelowy czas ujęcia musi mieścić się w zakresie 1–3 s.')
if SHOT_DURATION_TARGET not in {1, 2, 3}:
    raise ValueError('SHOT_DURATION_TARGET musi wynosić 1, 2 albo 3 sekundy.')

EPISODE_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
EPISODE_ID = (EPISODE_ID_OVERRIDE.strip() if EPISODE_ID_OVERRIDE else re.sub(r'[^a-z0-9]+', '-', f'lost-words-{EPISODE_STAMP}'.lower()).strip('-'))
FRAME_EPISODE_DIR = FRAMES_DIR / EPISODE_ID
SHOT_FRAME_DIR = FRAME_EPISODE_DIR / 'shots'
VIDEO_EPISODE_DIR = VIDEO_DIR / EPISODE_ID
SHOT_VIDEO_DIR = VISUALS_DIR / EPISODE_ID / 'shots'
CHECKPOINT_DIR = CHECKPOINTS_ROOT / EPISODE_ID
AUDIO_EPISODE_DIR = AUDIO_DIR / EPISODE_ID
STORY_PATH = ASSETS_DIR / f'{EPISODE_ID}_story.json'
SHOT_PLAN_PATH = ASSETS_DIR / f'{EPISODE_ID}_shot_manifest.json'
VISUAL_MANIFEST_PATH = ASSETS_DIR / f'{EPISODE_ID}_visual_manifest.json'
for folder in (FRAME_EPISODE_DIR, SHOT_FRAME_DIR, VIDEO_EPISODE_DIR, SHOT_VIDEO_DIR, CHECKPOINT_DIR, AUDIO_EPISODE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CHARACTER_BIBLE = {
    'survivor_1': (
        'fictional adult Polish male survivor, around 35 years old, tired hazel eyes, '
        'short dark hair, worn charcoal railway jacket, small old scar above eyebrow, '
        'consistent face and clothes in every shot'
    ),
    'survivor_2': (
        'fictional adult Polish female survivor, around 30 years old, pale tired face, '
        'dark hair tied back, faded olive coat, small analogue radio headset, '
        'consistent face and clothes in every shot'
    ),
    'unknown_entity': (
        'fictional adult unknown passenger, middle-aged, very pale face, wet dark coat, '
        'motionless unsettling expression, consistent appearance in every shot'
    ),
}

print(f'[OK] Seria: {SERIES_NAME} | temat: {EPISODE_TOPIC}')
print(f'[OK] Lokacja: {EPISODE_LOCATION} | sceny: {NUM_SCENES} | cel: {TARGET_SECONDS}s')
print(f'[INFO] Długość akceptowana: {MIN_VIDEO_SECONDS}–{MAX_VIDEO_SECONDS}s, cel minimum {GOAL_VIDEO_SECONDS}s')
print('[INFO] TEST_MODE:', TEST_MODE, '| lokalny renderer:', 'POMINIĘTY' if TEST_MODE else 'WŁĄCZONY')
print('[INFO] EPISODE_ID:', EPISODE_ID)
print('[INFO] Renderer:', VISUAL_RENDERER_PATH)
print('[INFO] OPENAI_API_KEY:', 'dostępny' if OPENAI_API_KEY else 'brak — scenariusz demo')


In [ ]:
# KOMÓRKA 3/7 — polski scenariusz dialogowy i walidacja JSON

ALLOWED_SPEAKERS = {'narrator', 'ocalaly_1', 'ocalaly_2', 'glos_z_radia', 'nieznany'}
ALLOWED_VOICE_ROLES = {'calm_dark', 'tense_male', 'calm_female', 'radio_distorted', 'unknown'}
ALLOWED_SHOT_TYPES = {'talking_character', 'environment'}


def strip_json_fence(raw_text):
    if not isinstance(raw_text, str) or not raw_text.strip():
        raise ValueError('Model zwrócił pustą odpowiedź.')
    fence = chr(96) * 3
    cleaned = re.sub(r'^\s*' + re.escape(fence) + r'(?:json)?\s*', '', raw_text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*' + re.escape(fence) + r'\s*$', '', cleaned)
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('Nie znaleziono kompletnego obiektu JSON.')
    return cleaned[start:end + 1]


def normalize_scene(item, label):
    if not isinstance(item, dict):
        raise TypeError(f'{label} musi być obiektem JSON.')
    required = (
        'scene_id', 'speaker', 'voice_role', 'text', 'visual_prompt', 'motion_prompt',
        'shot_type', 'character_id', 'character_description'
    )
    missing = [
        field for field in required
        if field not in item or not isinstance(item[field], str)
    ]
    if missing:
        raise ValueError(f'{label}: brak wymaganych pól: {", ".join(missing)}')

    scene_id = item['scene_id'].strip()
    if not re.fullmatch(r'[A-Za-z0-9_-]{1,20}', scene_id):
        raise ValueError(f'{label}: nieprawidłowe scene_id.')
    if item['speaker'] not in ALLOWED_SPEAKERS:
        raise ValueError(f'{label}: nieznany speaker={item["speaker"]!r}.')
    if item['voice_role'] not in ALLOWED_VOICE_ROLES:
        raise ValueError(f'{label}: nieznany voice_role={item["voice_role"]!r}.')
    if item['shot_type'] not in ALLOWED_SHOT_TYPES:
        raise ValueError(f'{label}: shot_type musi być talking_character albo environment.')

    spoken_text = re.sub(r'\s+', ' ', item['text']).strip()
    if not spoken_text:
        raise ValueError(f'{label}: text nie może być pusty.')
    if len(spoken_text.split()) > 24:
        raise ValueError(f'{label}: kwestia jest za długa — podziel ją na krótsze zdania.')
    if not re.search(r'[.!?…]$', spoken_text):
        spoken_text += '.'

    character_id = item['character_id'].strip() or 'none'
    character_description = item['character_description'].strip()
    if item['shot_type'] == 'talking_character':
        if item['speaker'] in {'narrator', 'glos_z_radia'}:
            raise ValueError(f'{label}: narrator i głos z radia powinny być ujęciem environment.')
        if not character_description:
            character_description = CHARACTER_BIBLE.get(character_id, '')
        if not character_description:
            raise ValueError(f'{label}: talking_character wymaga character_description.')
    if item['shot_type'] == 'environment' and item['speaker'] in {'narrator', 'glos_z_radia'}:
        character_id = 'none'

    visual_prompt = item['visual_prompt'].strip()
    motion_prompt = item['motion_prompt'].strip()
    if not motion_prompt:
        raise ValueError(f'{label}: motion_prompt nie może być pusty.')
    if item['shot_type'] == 'talking_character' and len(motion_prompt) < 25:
        raise ValueError(f'{label}: motion_prompt dla postaci musi opisywać konkretną akcję.')
    if item['shot_type'] == 'talking_character':
        visual_prompt += (
            ' Single fictional adult character, stable medium close-up, face and mouth clearly visible, '
            'front-facing or gentle three-quarter angle, lips relaxed, no cuts, no text in image.'
        )
    else:
        visual_prompt += ' Environmental survival-horror shot, no visible speaking face, no text in image.'

    return {
        'scene_id': scene_id,
        'speaker': item['speaker'],
        'voice_role': item['voice_role'],
        'text': spoken_text,
        'visual_prompt': visual_prompt,
        'motion_prompt': motion_prompt,
        'shot_type': item['shot_type'],
        'character_id': character_id,
        'character_description': character_description,
    }


def validate_story(payload):
    if not isinstance(payload, dict):
        raise TypeError('Scenariusz musi być obiektem JSON.')
    if not isinstance(payload.get('title'), str) or not payload['title'].strip():
        raise ValueError('Brak polskiego tytułu.')
    hook = normalize_scene(payload.get('hook'), 'hook')
    if len(hook['text'].split()) > 9:
        raise ValueError('Hook musi mieć maksymalnie 9 słów, aby zmieścił się w pierwszych 3 sekundach.')
    scenes = payload.get('scenes')
    if not isinstance(scenes, list) or len(scenes) != NUM_SCENES:
        raise ValueError(f'Wymagane jest dokładnie {NUM_SCENES} scen.')
    normalized, ids = [], {hook['scene_id']}
    for index, scene in enumerate(scenes, start=1):
        clean = normalize_scene(scene, f'scenes[{index-1}]')
        if clean['scene_id'] in ids:
            raise ValueError(f'Powtórzone scene_id: {clean["scene_id"]}')
        ids.add(clean['scene_id'])
        normalized.append(clean)
    talking_speakers = {
        item['speaker'] for item in [hook] + normalized
        if item['shot_type'] == 'talking_character'
    }
    if not talking_speakers:
        raise ValueError('Scenariusz musi zawierać co najmniej jedną widoczną mówiącą postać.')
    total_words = sum(len(item['text'].split()) for item in [hook] + normalized)
    if total_words < MIN_SCRIPT_WORDS:
        raise ValueError(f'Scenariusz ma {total_words} słów; wymagane minimum to {MIN_SCRIPT_WORDS}.')
    return {'title': payload['title'].strip(), 'hook': hook, 'scenes': normalized}


def fallback_story():
    # 12 rozbudowanych scen + krótki hook. Tekst jest celowo wystarczająco długi,
    # by polski TTS przekroczył 60 sekund bez zewnętrznego LLM.
    rows = [
        ('narrator','calm_dark',
         'Rano system alarmowy obudził całe miasto. Ewakuacja miała ruszyć o osiemnastej, lecz nikt nie wiedział, dokąd prowadzi ostatni pociąg.',
         'environment','none','',
         'Abandoned Polish city railway station at dawn, emergency loudspeaker above empty platform, wet concrete and distant smoke.',
         'The loudspeaker indicator flashes, loose evacuation notices tumble across the platform and dense fog rolls between the rails.'),
        ('ocalaly_1','tense_male',
         'Mapa wskazuje wejście od zachodu. Wczoraj sprawdzałem to miejsce: była tam ściana, bez drzwi i bez tunelu.',
         'talking_character','survivor_1',CHARACTER_BIBLE['survivor_1'],
         'Adult male survivor in a worn charcoal railway jacket inside a deserted station office, unfolded evacuation map on a desk, face and hands visible.',
         'He spreads the map with both hands, traces the western entrance, then looks toward the sealed doorway and takes one cautious step back.'),
        ('ocalaly_2','calm_female',
         'Zobacz ten atrament. Symbol jest świeży, ale papier pod nim wyblakł. Ktoś dopisał peron czwarty do starego planu.',
         'talking_character','survivor_2',CHARACTER_BIBLE['survivor_2'],
         'Adult female survivor in a faded olive coat leaning over a paper railway map, dusty desk, expressive face and hands visible.',
         'She leans over the desk, rubs the fresh ink with one finger, then turns the map toward her companion and points at the symbol.'),
        ('narrator','calm_dark',
         'Mieli jedną latarkę, radio i baterię na kilka minut. Na torach leżały mokre bilety z datą jutrzejszego dnia.',
         'environment','none','',
         'Survival supplies on a railway control desk: one flashlight, analogue radio, a nearly empty battery and damp paper tickets.',
         'A hand tests the flashlight, its beam flickers over tomorrow-dated tickets and the radio needle begins to shake.'),
        ('ocalaly_1','tense_male',
         'Ten peron zamknięto po wypadku. Nie było raportu, nazwisk ani zdjęć. Zniknął nawet z rozkładów jazdy.',
         'talking_character','survivor_1',CHARACTER_BIBLE['survivor_1'],
         'Adult male survivor at the far end of an abandoned railway platform, dark tracks behind him, cold mist and face clearly visible.',
         'He walks along a broken platform marking, stops abruptly, brushes weeds from the concrete and points toward a buried edge.'),
        ('glos_z_radia','radio_distorted',
         'Uwaga, mieszkańcy. Nie schodźcie pod ziemię. Jeśli usłyszycie własne nazwisko, nie odpowiadajcie i nie odwracajcie się.',
         'environment','none','',
         'Old analogue railway radio on a dusty metal desk, cracked speaker grille, red indicator lamp, no readable display.',
         'The speaker grille vibrates with the warning, the red lamp flashes irregularly and a loose cable swings against the desk.'),
        ('ocalaly_2','calm_female',
         'Ktoś stoi po drugiej stronie szyby. Nie rusza się, a jednak jego odbicie zbliża się do mnie.',
         'talking_character','survivor_2',CHARACTER_BIBLE['survivor_2'],
         'Adult female survivor beside a cracked station window, dim reflection of an unknown passenger behind the glass, face and hands visible.',
         'She notices the reflection, turns toward the empty platform, steps closer to the window and raises one hand to the glass.'),
        ('narrator','calm_dark',
         'Pod płytą peronu znaleźliśmy stalową klapę. Na metalu widniał numer cztery i data sprzed trzydziestu lat.',
         'environment','none','',
         'Hidden steel hatch beneath a broken railway platform slab, old bolts, rust, dust and a narrow flashlight beam.',
         'The flashlight sweeps over the hatch, grit falls from its edge and the metal shifts slightly as someone tests it.'),
        ('ocalaly_1','tense_male',
         'Nie otwieraj jej. Zegar pokazuje siedemnaście minut do ewakuacji, a z dołu słychać nadjeżdżający pociąg.',
         'talking_character','survivor_1',CHARACTER_BIBLE['survivor_1'],
         'Adult male survivor kneeling beside a hidden hatch, one hand raised to stop his companion, tense face lit by a flashlight.',
         'He raises his palm in warning, leans toward the hatch to listen, then turns sharply toward the station clock.'),
        ('nieznany','unknown',
         'Nie spóźniliście się. Pociąg czeka od dawna. Potrzebuję tylko osoby, której nazwisko wciąż widnieje na liście.',
         'talking_character','unknown_entity',CHARACTER_BIBLE['unknown_entity'],
         'Fictional adult unknown passenger in a dark railway maintenance corridor beyond a half-open hatch, pale face and wet dark coat.',
         'The passenger slowly steps out of the darkness, places one hand on the hatch frame and turns their face toward the survivors.'),
        ('ocalaly_2','calm_female',
         'To moje nazwisko. Nigdy tu nie byłam, a mimo to pamiętam każdy stopień tych schodów.',
         'talking_character','survivor_2',CHARACTER_BIBLE['survivor_2'],
         'Adult female survivor facing the unknown passenger in the underground corridor, pale practical light, face and mouth clearly visible.',
         'She freezes, looks down at a ticket in her hand, then takes one careful step toward the stairs as recognition crosses her face.'),
        ('narrator','calm_dark',
         'Na liście widniały trzy nazwiska. Dwa należały do ocalałych. Trzecie brzmiało tak samo jak nazwisko kobiety, ale miało starszą datę.',
         'environment','none','',
         'Abandoned evacuation checkpoint at dusk, clipboard with an old passenger list, two survivors reflected in dark glass, empty doorway.',
         'A page lifts in the wind as fresh ink slowly appears on the third line; the two reflections remain still while a third silhouette passes behind them.')
    ]

    scenes = []
    for index, row in enumerate(rows, start=1):
        speaker, role, text, shot_type, character_id, character_description, visual_prompt, motion_prompt = row
        scenes.append({
            'scene_id': f'S{index:02d}',
            'speaker': speaker,
            'voice_role': role,
            'text': text,
            'shot_type': shot_type,
            'character_id': character_id,
            'character_description': character_description,
            'visual_prompt': visual_prompt,
            'motion_prompt': motion_prompt,
        })

    hook = {
        'scene_id': 'H00',
        'speaker': 'glos_z_radia',
        'voice_role': 'radio_distorted',
        'text': 'Masz osiem godzin. Peron czwarty nie istnieje.',
        'visual_prompt': 'Abandoned railway platform at dawn, old loudspeaker and an unlit platform sign, no readable text.',
        'motion_prompt': 'A red warning lamp starts blinking above the empty platform as fog drifts over rails and loose paper crosses the frame.',
        'shot_type': 'environment',
        'character_id': 'none',
        'character_description': '',
    }
    return validate_story({'title': 'Peron, którego nie ma', 'hook': hook, 'scenes': scenes})


def generate_story_openai():
    if not OPENAI_API_KEY:
        raise RuntimeError('Brak OPENAI_API_KEY.')
    system_prompt = (
        'Jesteś scenarzystą filmowych, mrocznych polskich historii survival/apokalipsa w formacie short. '
        'Zwróć wyłącznie poprawny JSON. Tytuł, hook i wszystkie wypowiadane kwestie muszą być po polsku. '
        'Dialogi są krótkie, naturalne i przeplatane oszczędną narracją. '
        'W każdym odcinku pojawia się ktoś lub coś, czego nie powinno być w danej lokacji, '
        'oraz ukryty obiekt, pomieszczenie albo zapis nieobecny w żadnym wykazie. '
        'Klimat: filmowy scenariusz co-by-było-gdyby, deadline, ograniczone zasoby, rosnące konsekwencje i trudny wybór. ' 
        'Całość ma trwać 60–120 sekund; minimum 60 sekund, cel około 75 sekund. Unikaj krótkich historyjek. '
        'Bez romansów, humoru, lekkiego tonu i graficznej przemocy. '
        'Prompty visual_prompt, motion_prompt i character_description pisz po angielsku; wypowiedzi text wyłącznie po polsku. ' 
        'Każda scena ma mieć konkretną akcję: bohater chodzi, obraca się, gestykuluje, podnosi lub odkłada przedmiot albo reaguje na zdarzenie. Sam ruch kamery nie wystarcza.'
    )
    user_prompt = f"""Stwórz odcinek serii {SERIES_NAME}.
Temat: {EPISODE_TOPIC}
Lokacja: {EPISODE_LOCATION}
Styl scenariusza: {STORY_STYLE_PROFILE}
Styl wizualny: {VISUAL_STYLE}
Czas docelowy: około {TARGET_SECONDS} sekund; zakres {MIN_VIDEO_SECONDS}–{MAX_VIDEO_SECONDS} sekund. Minimum {MIN_SCRIPT_WORDS} wypowiadanych słów. Użyj 12 rozbudowanych scen.
Zwróć obiekt JSON z polskim title, obiektem hook i dokładnie {NUM_SCENES} elementami scenes.

Format każdej kwestii:
{{
  "scene_id": "H00 lub S01",
  "speaker": "narrator | ocalaly_1 | ocalaly_2 | glos_z_radia | nieznany",
  "voice_role": "calm_dark | tense_male | calm_female | radio_distorted | unknown",
  "text": "krótka naturalna wypowiedź po polsku",
  "visual_prompt": "English cinematic start-frame prompt, portrait 9:16, no text",
  "motion_prompt": "English description of one continuous visible physical action from first frame to end of clip",
  "shot_type": "talking_character albo environment",
  "character_id": "survivor_1, survivor_2, unknown_entity albo none",
  "character_description": "English stable adult character description; empty for environment"
}}

Dla widocznych wypowiedzi ocalałych/nieznanego użyj talking_character; twarz dorosłej fikcyjnej postaci ma być wyraźna, stabilna i front-facing. Dla narratora i głosu z radia użyj environment. Zachowuj ten sam character_id i opis postaci w kolejnych scenach. Hook ma mieć maksymalnie 9 słów i mocno otworzyć pierwsze 3 sekundy. Każdy text ma mieć naturalną interpunkcję. Każdy motion_prompt musi opisywać widoczny ruch postaci albo otoczenia, a nie tylko ruch kamery. Dla talking_character zachowaj twarz i usta w kadrze podczas ruchu głowy, rąk i tułowia. Nie dopisuj pól i nie zmieniaj liczby scen. Utrzymaj przyczynowo-skutkową fabułę, rosnącą stawkę i konsekwencję finałowej decyzji.
"""
    response = requests.post(
        f'{OPENAI_BASE_URL}/chat/completions',
        headers={'Authorization': f'Bearer {OPENAI_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': OPENAI_CHAT_MODEL,
            'response_format': {'type': 'json_object'},
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            'temperature': 0.85,
        },
        timeout=120,
    )
    response.raise_for_status()
    content = response.json()['choices'][0]['message']['content']
    return validate_story(json.loads(strip_json_fence(content)))


try:
    if not TEST_MODE and LLM_PROVIDER == 'openai_compatible' and OPENAI_API_KEY:
        STORY = generate_story_openai()
        print('[OK] Dynamiczny scenariusz wygenerowany przez model.')
    else:
        STORY = fallback_story()
        print('[INFO] Użyto polskiego scenariusza demonstracyjnego.')
except Exception as exc:
    print(f'[OSTRZEŻENIE] Generator scenariusza nie powiódł się: {type(exc).__name__}: {exc}')
    print('[INFO] Przechodzę na sprawdzony scenariusz demonstracyjny.')
    STORY = fallback_story()

SEGMENT_SPECS = [STORY['hook']] + STORY['scenes']
STORY_PATH = ASSETS_DIR / f'{EPISODE_ID}_story.json'
write_json_atomic(STORY_PATH, STORY)

print('\nTYTUŁ:', STORY['title'])
for item in SEGMENT_SPECS:
    print(
        f"{item['scene_id']} | {item['speaker']} | {item['shot_type']} | "
        f"{item['voice_role']}: {item['text']}"
    )
print(f'[OK] Polski scenariusz JSON: {STORY_PATH}')


In [ ]:
# KOMÓRKA 4/7 — osobne pliki TTS dla każdej kwestii

import asyncio

VOICE_PROFILES = {
    'calm_dark': {'voice': 'pl-PL-MarekNeural', 'rate': '-5%', 'pitch': '-7Hz'},
    'tense_male': {'voice': 'pl-PL-MarekNeural', 'rate': '+1%', 'pitch': '-9Hz'},
    'calm_female': {'voice': 'pl-PL-ZofiaNeural', 'rate': '-3%', 'pitch': '-4Hz'},
    'radio_distorted': {'voice': 'pl-PL-AgnieszkaNeural', 'rate': '-6%', 'pitch': '-10Hz'},
    'unknown': {'voice': 'pl-PL-MarekNeural', 'rate': '-8%', 'pitch': '-12Hz'},
}

SPEAKER_LABELS_PL = {
    'narrator': 'DZIENNIK',
    'ocalaly_1': 'OCALAŁY 1',
    'ocalaly_2': 'OCALAŁA 2',
    'glos_z_radia': 'GŁOS Z RADIA',
    'nieznany': 'NIEZNANY',
}


def normalize_edge_tts_adjustment(value, unit):
    """Normalizuje rate/pitch do formatu wymaganego przez Edge TTS; wartości neutralne pomija."""
    if value is None:
        return None

    raw = str(value).strip()
    if raw.casefold() in {'', 'default', '0', '0%', '0hz'}:
        return None

    match = re.fullmatch(r'([+-]?\d+(?:[.,]\d+)?)\s*(%|hz)?', raw, flags=re.IGNORECASE)
    if not match:
        raise ValueError(
            f"Niepoprawna wartość parametru Edge TTS: {value!r}; oczekiwano liczby z jednostką {unit}."
        )

    number = float(match.group(1).replace(',', '.'))
    if number == 0:
        return None

    supplied_unit = match.group(2)
    if supplied_unit and supplied_unit.casefold() != unit.casefold():
        raise ValueError(
            f"Niepoprawna jednostka Edge TTS: {value!r}; dla tego parametru wymagana jest jednostka {unit}."
        )

    limits = {'%': (-25.0, 25.0), 'Hz': (-20.0, 20.0)}
    lower, upper = limits[unit]
    if not lower <= number <= upper:
        raise ValueError(
            f"Parametr {unit}={number:g} poza bezpiecznym zakresem {lower:g}…{upper:g}."
        )

    sign = '+' if number > 0 else '-'
    return f"{sign}{abs(number):g}{unit}"


# Szybki test normalizatora — bez połączenia z usługą TTS.
assert normalize_edge_tts_adjustment(None, '%') is None
assert normalize_edge_tts_adjustment('default', 'Hz') is None
assert normalize_edge_tts_adjustment('0%', '%') is None
assert normalize_edge_tts_adjustment('0Hz', 'Hz') is None
assert normalize_edge_tts_adjustment('5', '%') == '+5%'
assert normalize_edge_tts_adjustment('-5Hz', 'Hz') == '-5Hz'


def _is_retryable_edge_tts_error(exc):
    """Rozpoznaje brak audio i typowe przejściowe błędy sieciowe."""
    error_name = type(exc).__name__.casefold()
    error_text = str(exc).casefold()

    if error_name == 'noaudioreceived':
        return True
    if isinstance(exc, (asyncio.TimeoutError, TimeoutError, ConnectionError, OSError)):
        return True

    try:
        import aiohttp
        if isinstance(exc, aiohttp.ClientError):
            return True
    except Exception:
        pass

    transient_markers = (
        'no audio was received',
        'temporarily unavailable',
        'server disconnected',
        'connection reset',
        'connection aborted',
        'connection refused',
        'timed out',
        'timeout',
        'network is unreachable',
        'name or service not known',
        '429', '502', '503', '504',
    )
    return any(marker in error_text for marker in transient_markers)


async def create_edge_tts_segment(item, mp3_path):
    scene_id = str(item.get('scene_id') or '<brak scene_id>')
    voice_role = item.get('voice_role')
    profile = VOICE_PROFILES.get(voice_role)
    if not profile:
        raise ValueError(f"{scene_id}: nieznany voice_role={voice_role!r}.")

    voice = profile.get('voice')
    clean_text = str(item.get('text') or '').strip()
    if not clean_text or not any(char.isalnum() for char in clean_text):
        raise ValueError(
            f"{scene_id}: tekst dla Edge TTS jest pusty lub zawiera wyłącznie interpunkcję/znaki specjalne."
        )

    # Parametry neutralne pomijamy całkowicie — nie wysyłamy np. rate='0%' ani pitch='0Hz'.
    try:
        normalized_rate = normalize_edge_tts_adjustment(profile.get('rate'), '%')
        normalized_pitch = normalize_edge_tts_adjustment(profile.get('pitch'), 'Hz')
    except ValueError as exc:
        raise ValueError(f"{scene_id}: niepoprawne ustawienia głosu {voice!r}: {exc}") from exc

    try:
        import edge_tts
    except ImportError as exc:
        raise RuntimeError(
            "Brak biblioteki edge-tts. Uruchom komórkę instalacyjną przed komórką TTS."
        ) from exc

    tts_kwargs = {
        'text': clean_text,
        'voice': voice,
        'volume': '+0%',
    }
    if normalized_rate is not None:
        tts_kwargs['rate'] = normalized_rate
    if normalized_pitch is not None:
        tts_kwargs['pitch'] = normalized_pitch

    mp3_path = Path(mp3_path)
    max_attempts = 3
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            # Usuwamy ewentualny plik częściowy z poprzedniej próby.
            if mp3_path.exists():
                mp3_path.unlink()
            mp3_path.parent.mkdir(parents=True, exist_ok=True)

            communicate = edge_tts.Communicate(**tts_kwargs)
            await communicate.save(str(mp3_path))

            if not mp3_path.exists() or mp3_path.stat().st_size < 1024:
                raise RuntimeError('Edge TTS nie zapisał poprawnego audio lub zwrócił pusty/niepełny plik MP3.')

            return mp3_path

        except Exception as exc:
            last_error = exc
            try:
                if mp3_path.exists():
                    mp3_path.unlink()
            except OSError:
                pass

            if attempt >= max_attempts or not _is_retryable_edge_tts_error(exc):
                break

            delay_seconds = min(1.0 * attempt, 3.0)
            print(
                f"[TTS RETRY] {scene_id}, próba {attempt}/{max_attempts}, "
                f"{type(exc).__name__}; ponawiam za {delay_seconds:.1f}s."
            )
            await asyncio.sleep(delay_seconds)

    # Podgląd tekstu nie zawiera żadnych sekretów/parametrów połączenia.
    preview = re.sub(r'\s+', ' ', clean_text[:100]).strip()
    if len(clean_text) > 100:
        preview += '…'
    raise RuntimeError(
        "Edge TTS nie wygenerował poprawnego MP3 po "
        f"{attempt} próbach | scene_id={scene_id!r} | voice={voice!r} | "
        f"rate={normalized_rate!r} | pitch={normalized_pitch!r} | "
        f"długość_tekstu={len(clean_text)} | tekst_początek={preview!r} | "
        f"błąd={type(last_error).__name__ if last_error else 'UnknownError'}."
    ) from last_error

def convert_tts_to_wav(mp3_path, wav_path, item):
    is_radio = (
        item['voice_role'] in {'radio_distorted', 'unknown'}
        or item['speaker'] in {'glos_z_radia', 'nieznany'}
    )
    if is_radio:
        filters = (
            '[0:a]aresample=48000,highpass=f=240,lowpass=f=3600,'
            'acompressor=threshold=0.09:ratio=2.4:attack=12:release=130[voice];'
            '[1:a]highpass=f=850,lowpass=f=4200,'
            f'volume={RADIO_STATIC_GAIN:.3f}[static];'
            '[voice][static]amix=inputs=2:duration=first:normalize=0,alimiter=limit=0.95[out]'
        )
        run_cmd([
            'ffmpeg', '-y', '-i', str(mp3_path),
            '-f', 'lavfi', '-i', 'anoisesrc=color=pink:amplitude=0.008:sample_rate=48000',
            '-filter_complex', filters, '-map', '[out]',
            '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(wav_path)
        ])
    else:
        run_cmd([
            'ffmpeg', '-y', '-i', str(mp3_path),
            '-af', 'aresample=48000,highpass=f=70,lowpass=f=16000,'
                   'acompressor=threshold=0.10:ratio=2.0:attack=18:release=220,'
                   'alimiter=limit=0.95',
            '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(wav_path)
        ])
    if not wav_path.exists() or wav_path.stat().st_size < 4000:
        raise RuntimeError(f'Nie powstał poprawny plik audio {wav_path.name}.')
    media = probe_media(wav_path)
    if media['audio'] is None or media['video'] is not None:
        raise RuntimeError(f'Plik {wav_path.name} nie zawiera prawidłowego, samodzielnego strumienia audio.')
    if probe_duration(wav_path) < 0.20:
        raise RuntimeError(f'Plik audio {wav_path.name} jest podejrzanie krótki.')


if TTS_PROVIDER != 'edge':
    raise NotImplementedError(
        'TTS_PROVIDER=external jest miejscem na adapter zewnętrzny; domyślny, gotowy adapter to edge-tts.'
    )

# Ustaw na "H00", aby przetestować wyłącznie problematyczny segment.
# Przed pełnym renderem przywróć None.
TTS_TEST_SCENE_ID = None
TTS_ITEMS_TO_PROCESS = [
    item for item in SEGMENT_SPECS
    if TTS_TEST_SCENE_ID is None or item['scene_id'] == TTS_TEST_SCENE_ID
]
if TTS_TEST_SCENE_ID is not None and not TTS_ITEMS_TO_PROCESS:
    raise ValueError(f"Nie znaleziono segmentu TTS o scene_id={TTS_TEST_SCENE_ID!r}.")

TIMELINE = []
cursor = 0.0
for index, item in enumerate(TTS_ITEMS_TO_PROCESS):
    mp3_path = AUDIO_EPISODE_DIR / f"{item['scene_id']}.mp3"
    wav_path = AUDIO_EPISODE_DIR / f"{item['scene_id']}.wav"
    await create_edge_tts_segment(item, mp3_path)
    if not mp3_path.exists() or mp3_path.stat().st_size < 1000:
        raise RuntimeError(f'TTS zwrócił pusty plik: {mp3_path.name}')
    convert_tts_to_wav(mp3_path, wav_path, item)
    duration = probe_duration(wav_path)
    pause = PAUSE_BETWEEN_LINES if index < len(TTS_ITEMS_TO_PROCESS) - 1 else 0.0
    TIMELINE.append({
        'scene_id': item['scene_id'],
        'audio_path': str(wav_path),
        'duration': duration,
        'start': cursor,
        'pause_after': pause,
        'speaker': item['speaker'],
        'text': item['text'],
    })
    cursor += duration + pause
    print(
        f"[OK] {item['scene_id']} | {SPEAKER_LABELS_PL[item['speaker']]} | "
        f"{VOICE_PROFILES[item['voice_role']]['voice']} | {duration:.2f}s"
    )

TOTAL_DIALOGUE_SECONDS = cursor
write_json_atomic(AUDIO_EPISODE_DIR / 'timeline.json', TIMELINE)
print(f'[OK] Oddzielne pliki audio: {len(TIMELINE)}')
print(f'[INFO] Czas dialogów z pauzami: {TOTAL_DIALOGUE_SECONDS:.2f}s')
if TTS_TEST_SCENE_ID is None:
    if TOTAL_DIALOGUE_SECONDS < MIN_VIDEO_SECONDS:
        raise ValueError(f'Audio ma {TOTAL_DIALOGUE_SECONDS:.1f}s; minimum projektu to {MIN_VIDEO_SECONDS}s. Rozbuduj polskie kwestie w fallback_story lub zmień tempo narracji.')
    if TOTAL_DIALOGUE_SECONDS > MAX_VIDEO_SECONDS:
        raise ValueError(f'Audio ma {TOTAL_DIALOGUE_SECONDS:.1f}s; maksimum projektu to {MAX_VIDEO_SECONDS}s. Skróć scenariusz.')
    if TOTAL_DIALOGUE_SECONDS < GOAL_VIDEO_SECONDS:
        print(f'[OSTRZEŻENIE] Materiał ma mniej niż cel {GOAL_VIDEO_SECONDS}s; rozważ dodanie scen lub narracji.')
    elif TOTAL_DIALOGUE_SECONDS >= GOAL_VIDEO_SECONDS:
        print(f'[OK] Osiągnięto cel długości co najmniej {GOAL_VIDEO_SECONDS}s.')


In [ ]:
# KOMÓRKA 5/7 — manifest krótkich ujęć 1–3 s i fragmenty audio dla renderera Claude

SHOT_AUDIO_DIR = AUDIO_EPISODE_DIR / 'shots'
SHOT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

CHARACTER_SHOT_KINDS = [
    ('medium_action', 'Medium action shot: show the same character performing the action; face, mouth, shoulders and hands visible.'),
    ('insert_detail', 'Insert detail: show the character hands and the important object moving; preserve same location and props.'),
    ('side_follow', 'Side follow shot: show the same character turning, stepping or shifting position; visible body movement.'),
    ('reaction', 'Reaction shot: return to the same character face; show a clear emotional response and keep the mouth unobstructed.'),
    ('over_shoulder', 'Over-the-shoulder shot: show the character reaching or pointing toward the object; preserve a readable body silhouette.'),
    ('wide_follow', 'Wider follow-through shot: show the character completing the action with a step or gesture; actor movement leads.')
]
ENVIRONMENT_SHOT_KINDS = [
    ('wide', 'Wide establishing shot: reveal the location with moving atmosphere and visible depth.'),
    ('insert_detail', 'Insert detail: close on a meaningful object, paper, radio, clock or light; show it moving or reacting.'),
    ('tracking', 'Lateral tracking shot: move through the location and reveal fog, rain, dust or practical lights.'),
    ('reveal', 'Suspense reveal shot: reveal a doorway, hidden area, approaching light or unexpected silhouette.'),
    ('low_detail', 'Low angle detail: show moving foreground elements and depth into the location.')
]


def split_duration_into_shots(total_seconds, target=SHOT_DURATION_TARGET,
                              minimum=SHOT_DURATION_MIN, maximum=SHOT_DURATION_MAX):
    """Dzieli długość wypowiedzi na równe ujęcia 1–3 s, bez końcowego mikro-ujęcia."""
    total = float(total_seconds)
    if total <= 0:
        raise ValueError(f'Niepoprawna długość sceny: {total_seconds}.')
    count = max(1, int(math.ceil(total / target)))
    while count > 1 and total / count < minimum:
        count -= 1
    while total / count > maximum:
        count += 1
    durations = [round(total / count, 3) for _ in range(count)]
    durations[-1] = round(total - sum(durations[:-1]), 3)
    if any(value < minimum - 0.01 or value > maximum + 0.01 for value in durations):
        raise ValueError(
            f'Nie można podzielić {total:.3f}s na ujęcia {minimum:.1f}–{maximum:.1f}s: {durations}'
        )
    if abs(sum(durations) - total) > 0.01:
        raise RuntimeError(f'Suma czasu ujęć {durations} nie odpowiada długości sceny {total:.3f}s.')
    return durations


SHOT_PLAN = []
shot_order = 0
timeline_by_scene = {entry['scene_id']: entry for entry in TIMELINE}

for item in SEGMENT_SPECS:
    timeline = timeline_by_scene.get(item['scene_id'])
    if timeline is None:
        raise RuntimeError(f"Brak audio w TIMELINE dla {item['scene_id']}.")
    source_audio = Path(timeline['audio_path'])
    if not source_audio.exists() or source_audio.stat().st_size < 4000:
        raise RuntimeError(f"{item['scene_id']}: brak pełnego pliku WAV {source_audio}.")
    source_media = probe_media(source_audio)
    if source_media['audio'] is None or source_media['video'] is not None:
        raise RuntimeError(f"{item['scene_id']}: plik źródłowy nie jest poprawnym WAV audio.")
    measured_source_duration = probe_duration(source_audio)
    if abs(measured_source_duration - float(timeline['duration'])) > 0.15:
        raise RuntimeError(
            f"{item['scene_id']}: długość WAV {measured_source_duration:.2f}s "
            f"nie zgadza się z timeline {timeline['duration']:.2f}s."
        )

    durations = split_duration_into_shots(timeline['duration'])
    templates = CHARACTER_SHOT_KINDS if item['shot_type'] == 'talking_character' else ENVIRONMENT_SHOT_KINDS
    local_offset = 0.0

    for shot_index, duration in enumerate(durations, start=1):
        shot_kind, shot_direction = templates[(shot_index - 1) % len(templates)]
        shot_id = f"{item['scene_id']}_SH{shot_index:02d}"
        frame_path = SHOT_FRAME_DIR / f'{shot_id}.png'
        output_path = SHOT_VIDEO_DIR / f'{shot_id}.mp4'
        checkpoint_path = CHECKPOINT_DIR / f'{shot_id}.json'
        audio_clip_path = SHOT_AUDIO_DIR / f'{shot_id}.wav'

        visual_prompt = (
            f"{item['visual_prompt']} {item['character_description']} "
            f"{shot_direction} Cinematic portrait 9:16, realistic lighting, no text, no logo."
        )
        motion_prompt = (
            f"{item['motion_prompt']} {shot_direction} "
            "One continuous physical action with clear subject movement, coherent anatomy and believable physics. "
            "Keep character identity, clothing, props and location consistent."
        )
        if item['shot_type'] == 'talking_character' and shot_kind in {'medium_action', 'reaction', 'side_follow'}:
            visual_prompt += " Keep the adult character face and mouth visible for this dialogue cut."

        shot = {
            'schema_version': '1.0',
            'episode_id': EPISODE_ID,
            'shot_id': shot_id,
            'scene_id': item['scene_id'],
            'order': shot_order,
            'timeline_start_sec': round(float(timeline['start']) + local_offset, 3),
            'duration_sec': duration,
            'shot_index': shot_index,
            'shot_count_in_scene': len(durations),
            'shot_type': item['shot_type'],
            'shot_kind': shot_kind,
            'visual_prompt_en': visual_prompt,
            'motion_prompt_en': motion_prompt,
            'speaker': item['speaker'],
            'voice_role': item['voice_role'],
            'character_id': item['character_id'],
            'character_description_en': item['character_description'],
            'audio_path': timeline['audio_path'],
            'audio_offset_sec': round(local_offset, 3),
            'audio_duration_sec': duration,
            'audio_clip_path': str(audio_clip_path),
            'lipsync_recommended': (
                item['shot_type'] == 'talking_character'
                and shot_kind in {'medium_action', 'reaction'}
            ),
            'frame_path': str(frame_path),
            'output_path': str(output_path),
            'checkpoint_path': str(checkpoint_path),
            'output_width': VIDEO_WIDTH,
            'output_height': VIDEO_HEIGHT,
            'output_fps': FPS,
        }

        # Wytnij ten dokładny fragment WAV, aby lokalny renderer mógł opcjonalnie
        # użyć go do synchronizacji ust bez usług płatnych.
        run_cmd([
            'ffmpeg', '-y', '-ss', f"{local_offset:.3f}",
            '-i', str(timeline['audio_path']),
            '-t', f'{duration:.3f}',
            '-vn', '-ar', '48000', '-ac', '2',
            '-c:a', 'pcm_s16le', str(audio_clip_path)
        ])
        if not audio_clip_path.exists() or audio_clip_path.stat().st_size < 2000:
            raise RuntimeError(f'Nie powstał fragment audio dla {shot_id}.')
        clip_media = probe_media(audio_clip_path)
        clip_duration = probe_duration(audio_clip_path)
        if clip_media['audio'] is None or clip_media['video'] is not None:
            raise RuntimeError(f'{shot_id}: crop nie zawiera prawidłowego strumienia audio.')
        if abs(clip_duration - duration) > 0.15:
            raise RuntimeError(
                f'{shot_id}: fragment audio ma {clip_duration:.2f}s, oczekiwano {duration:.2f}s.'
            )

        SHOT_PLAN.append(shot)
        shot_order += 1
        local_offset += duration

write_json_atomic(SHOT_PLAN_PATH, {
    'schema_version': '1.0',
    'episode_id': EPISODE_ID,
    'target_seconds': TARGET_SECONDS,
    'min_accepted_seconds': MIN_VIDEO_SECONDS,
    'goal_seconds': GOAL_VIDEO_SECONDS,
    'max_seconds': MAX_VIDEO_SECONDS,
    'target_shot_seconds': SHOT_DURATION_TARGET,
    'min_shot_seconds': SHOT_DURATION_MIN,
    'max_shot_seconds': SHOT_DURATION_MAX,
    'render_size': 'selected dynamically in cell 6 from available VRAM',
    'output_size': [VIDEO_WIDTH, VIDEO_HEIGHT],
    'fps': FPS,
    'shots': SHOT_PLAN,
})

shot_durations = [shot['duration_sec'] for shot in SHOT_PLAN]
print(f'[OK] Zapisano manifest: {SHOT_PLAN_PATH}')
print(f'[OK] Liczba krótkich ujęć: {len(SHOT_PLAN)}')
print(f'[INFO] Zakres czasu ujęć: {min(shot_durations):.2f}–{max(shot_durations):.2f}s')
print(f'[INFO] Łączny czas obrazu: {sum(shot_durations):.2f}s')


In [ ]:
# KOMÓRKA 6/7 — lokalny renderer Claude, T4/CPU, kontrola VRAM i checkpoint każdego ujęcia

VISUAL_RECORDS = []


def detect_render_runtime():
    """Preferuje CUDA/T4, dynamicznie dobiera pionową rozdzielczość; CPU jako fallback."""
    info = {
        'device': 'cpu',
        'device_name': 'CPU',
        'cuda_available': False,
        'vram_total_gb': 0.0,
        'vram_free_gb': 0.0,
        'render_width': 288,
        'render_height': 512,
        'output_width': VIDEO_WIDTH,
        'output_height': VIDEO_HEIGHT,
        'output_fps': FPS,
        'batch_size': 1,
        'dtype': 'float32',
        'max_vram_gb': 0.0,
        'enable_cpu_offload': False,
        'attention_slicing': True,
        'vae_slicing': True,
        'decode_chunk_size': 2,
        'model_cache_dir': str(MODELS_DIR),
        'checkpoint_dir': str(CHECKPOINT_DIR),
        'local_only_inference': True,
        'allow_paid_apis': False,
    }
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            free_bytes, total_bytes = torch.cuda.mem_get_info(0)
            total_gb = total_bytes / (1024 ** 3)
            free_gb = free_bytes / (1024 ** 3)
            name = torch.cuda.get_device_name(0)
            info.update({
                'device': 'cuda:0',
                'device_name': name,
                'cuda_available': True,
                'vram_total_gb': round(total_gb, 2),
                'vram_free_gb': round(free_gb, 2),
                'dtype': 'float16',
                'max_vram_gb': round(max(0.0, min(12.0, free_gb - 2.0)), 2),
                'enable_cpu_offload': free_gb < 13.0,
            })
            # Gdy wolna VRAM spada poniżej 3 GB, wybierz CPU zamiast przekraczać budżet.
            if free_gb < 3.0:
                info.update({
                    'device': 'cpu',
                    'device_name': f'{name} detected; CPU fallback (<3 GB free VRAM)',
                    'cuda_available': False,
                    'dtype': 'float32',
                    'max_vram_gb': 0.0,
                    'enable_cpu_offload': False,
                    'render_width': 288,
                    'render_height': 512,
                })
            elif free_gb >= 11.0:
                info.update({'render_width': 432, 'render_height': 768})
            elif free_gb >= 7.0:
                info.update({'render_width': 360, 'render_height': 640})
            else:
                info.update({'render_width': 288, 'render_height': 512})
        else:
            print('[INFO] CUDA niedostępne — renderer przejdzie na CPU. TPU nie jest używany w tym profilu.')
    except ImportError:
        print('[INFO] PyTorch niedostępny — renderer musi użyć ścieżki CPU.')
    except Exception as exc:
        print(f'[OSTRZEŻENIE] Nie udało się odczytać GPU/VRAM; wybieram CPU: {exc}')

    info['max_render_pixels'] = info['render_width'] * info['render_height']
    return info


def refresh_runtime_for_shot(base_runtime):
    """Odświeża wolną VRAM tuż przed generowaniem każdego ujęcia."""
    current = dict(base_runtime)
    if not current.get('cuda_available'):
        return current
    try:
        import torch
        if not torch.cuda.is_available():
            current.update({
                'device': 'cpu',
                'device_name': 'CPU fallback; CUDA no longer available',
                'cuda_available': False,
                'vram_free_gb': 0.0,
                'max_vram_gb': 0.0,
                'dtype': 'float32',
                'render_width': 288,
                'render_height': 512,
                'enable_cpu_offload': False,
            })
        else:
            free_bytes, total_bytes = torch.cuda.mem_get_info(0)
            free_gb = free_bytes / (1024 ** 3)
            total_gb = total_bytes / (1024 ** 3)
            current['vram_free_gb'] = round(free_gb, 2)
            current['vram_total_gb'] = round(total_gb, 2)
            current['max_vram_gb'] = round(max(0.0, min(12.0, free_gb - 2.0)), 2)
            current['enable_cpu_offload'] = free_gb < 13.0
            if free_gb < 3.0:
                current.update({
                    'device': 'cpu',
                    'device_name': f"{current['device_name']}; CPU fallback (<3 GB free VRAM)",
                    'cuda_available': False,
                    'dtype': 'float32',
                    'max_vram_gb': 0.0,
                    'enable_cpu_offload': False,
                    'render_width': 288,
                    'render_height': 512,
                })
            elif free_gb >= 11.0:
                current.update({'render_width': 432, 'render_height': 768})
            elif free_gb >= 7.0:
                current.update({'render_width': 360, 'render_height': 640})
            else:
                current.update({'render_width': 288, 'render_height': 512})
    except Exception as exc:
        print(f"[OSTRZEŻENIE] Nie udało się odświeżyć VRAM; zachowuję profil bazowy: {exc}")
    current['max_render_pixels'] = current['render_width'] * current['render_height']
    current['batch_size'] = 1
    return current


RENDER_RUNTIME = detect_render_runtime()
for shot in SHOT_PLAN:
    shot['render_width'] = RENDER_RUNTIME['render_width']
    shot['render_height'] = RENDER_RUNTIME['render_height']

print(
    f"[RUNTIME] {RENDER_RUNTIME['device_name']} | {RENDER_RUNTIME['device']} | "
    f"VRAM {RENDER_RUNTIME['vram_free_gb']:.1f}/{RENDER_RUNTIME['vram_total_gb']:.1f} GB | "
    f"render {RENDER_RUNTIME['render_width']}x{RENDER_RUNTIME['render_height']} | final {VIDEO_WIDTH}x{VIDEO_HEIGHT}"
)
write_json_atomic(SHOT_PLAN_PATH, {
    'schema_version': '1.0',
    'episode_id': EPISODE_ID,
    'runtime': RENDER_RUNTIME,
    'shots': SHOT_PLAN,
})

if TEST_MODE:
    print('[TEST_MODE] Manifest ujęć i fragmenty audio gotowe.')
    print('[TEST_MODE] Pominięto lokalny renderer; nie powstanie MP4.')
else:
    if not VISUAL_RENDERER_PATH.exists():
        print('[INFO] Brak modułu na Drive. Pobieram visuals/shot_renderer.py z repozytorium.')
        response = requests.get(VISUAL_RENDERER_SOURCE_URL, timeout=60)
        if response.status_code == 404:
            raise FileNotFoundError(
                'visuals/shot_renderer.py nie ma jeszcze w głównej gałęzi repozytorium. '
                'Claude musi dodać moduł i połączyć jego PR do main przed pełnym renderowaniem.'
            )
        response.raise_for_status()
        VISUAL_RENDERER_PATH.parent.mkdir(parents=True, exist_ok=True)
        VISUAL_RENDERER_PATH.write_text(response.text, encoding='utf-8')

    if not VISUAL_RENDERER_PATH.exists() or VISUAL_RENDERER_PATH.stat().st_size < 1000:
        raise RuntimeError(
            f'Brak gotowego modułu {VISUAL_RENDERER_PATH}. '
            'Claude musi dodać visuals/shot_renderer.py zgodnie z docs/visual_renderer_contract.md.'
        )

    spec = importlib.util.spec_from_file_location('lost_words_shot_renderer', VISUAL_RENDERER_PATH)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Nie można załadować modułu renderera: {VISUAL_RENDERER_PATH}')
    renderer_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(renderer_module)
    if not callable(getattr(renderer_module, 'render_shot', None)):
        raise RuntimeError('Moduł Claude musi eksportować funkcję render_shot(shot: dict, runtime: dict) -> dict.')

    for shot in SHOT_PLAN:
        output_path = Path(shot['output_path'])
        checkpoint_path = Path(shot['checkpoint_path'])
        checkpoint = read_json_if_exists(checkpoint_path, default={})

        # Resume: poprawne MP4 + completed JSON = pomiń ponowne generowanie.
        if (
            RESUME_RENDERING
            and checkpoint.get('status') == 'completed'
            and checkpoint.get('shot_id') == shot['shot_id']
            and output_path.exists()
        ):
            try:
                media = probe_media(output_path)
                duration = probe_duration(output_path)
                width = int(media['video'].get('width') or 0) if media['video'] else 0
                height = int(media['video'].get('height') or 0) if media['video'] else 0
                portrait_ok = width > 0 and height > 0 and abs((width / height) - (9 / 16)) <= 0.025
                duration_ok = abs(duration - float(shot['duration_sec'])) <= 0.20
                silent_ok = media['audio'] is None
                if media['video'] and portrait_ok and duration_ok and silent_ok:
                    record = dict(checkpoint)
                    record.update({
                        'shot_id': shot['shot_id'],
                        'scene_id': shot['scene_id'],
                        'status': 'completed',
                        'video_path': str(output_path),
                        'duration_sec': duration,
                        'width': width,
                        'height': height,
                        'checkpoint_path': str(checkpoint_path),
                    })
                    VISUAL_RECORDS.append(record)
                    print(f"[RESUME] {shot['shot_id']} — checkpoint poprawny, pomijam.")
                    continue
            except Exception:
                pass

        print(
            f"[RENDER] {shot['order'] + 1}/{len(SHOT_PLAN)} {shot['shot_id']} "
            f"({shot['duration_sec']:.2f}s, {shot['shot_kind']})"
        )
        # Stopniowo zmniejsz rozdzielczość po CUDA OOM; ostatnia próba używa CPU.
        current_runtime = refresh_runtime_for_shot(RENDER_RUNTIME)
        runtime_candidates = [dict(current_runtime)]
        if current_runtime['cuda_available']:
            for width, height in [(360, 640), (288, 512)]:
                if width < current_runtime['render_width']:
                    lower = dict(current_runtime)
                    lower.update({
                        'render_width': width,
                        'render_height': height,
                        'max_render_pixels': width * height,
                        'enable_cpu_offload': True,
                    })
                    runtime_candidates.append(lower)
            cpu_fallback = dict(current_runtime)
            cpu_fallback.update({
                'device': 'cpu',
                'device_name': 'CPU fallback after CUDA OOM',
                'cuda_available': False,
                'vram_total_gb': 0.0,
                'vram_free_gb': 0.0,
                'render_width': 288,
                'render_height': 512,
                'max_render_pixels': 288 * 512,
                'dtype': 'float32',
                'max_vram_gb': 0.0,
                'enable_cpu_offload': False,
            })
            runtime_candidates.append(cpu_fallback)

        result = None
        runtime_used = None
        last_oom = None
        for attempt_index, runtime_candidate in enumerate(runtime_candidates, start=1):
            shot['render_width'] = runtime_candidate['render_width']
            shot['render_height'] = runtime_candidate['render_height']
            try:
                result = renderer_module.render_shot(shot, runtime_candidate)
                runtime_used = runtime_candidate
                break
            except Exception as exc:
                is_cuda_oom = (
                    runtime_candidate.get('cuda_available')
                    and ('out of memory' in str(exc).casefold() or 'cuda oom' in str(exc).casefold())
                )
                if not is_cuda_oom or attempt_index >= len(runtime_candidates):
                    raise
                last_oom = exc
                print(
                    f"[VRAM] {shot['shot_id']}: CUDA OOM przy "
                    f"{runtime_candidate['render_width']}x{runtime_candidate['render_height']}; "
                    'czyszczę cache i próbuję niższego profilu.'
                )
                try:
                    import torch
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                try:
                    partial_output = Path(shot['output_path'])
                    if partial_output.exists():
                        partial_output.unlink()
                except OSError:
                    pass

        if result is None:
            raise RuntimeError(f"{shot['shot_id']}: render nie powiódł się po fallbackach sprzętowych: {last_oom}")
        if not isinstance(result, dict):
            raise TypeError(f"{shot['shot_id']}: render_shot musi zwrócić dict.")
        if result.get('status') != 'completed':
            raise RuntimeError(f"{shot['shot_id']}: renderer zwrócił status {result.get('status')!r}.")
        if result.get('shot_id') != shot['shot_id']:
            raise RuntimeError(
                f"{shot['shot_id']}: renderer zwrócił shot_id={result.get('shot_id')!r}; "
                'identyfikator musi zgadzać się z manifestem.'
            )
        video_path = Path(result.get('video_path') or shot['output_path'])
        if not video_path.exists() or video_path.stat().st_size < 10000:
            raise RuntimeError(f"{shot['shot_id']}: renderer nie zapisał poprawnego MP4: {video_path}")
        media = probe_media(video_path)
        if media['video'] is None:
            raise RuntimeError(f"{shot['shot_id']}: wynik nie zawiera strumienia video.")
        if media['audio'] is not None:
            raise RuntimeError(
                f"{shot['shot_id']}: renderer ma zwracać cichy MP4; wykryto nieoczekiwany strumień audio."
            )
        measured_duration = probe_duration(video_path)
        if abs(measured_duration - float(shot['duration_sec'])) > 0.20:
            raise RuntimeError(
                f"{shot['shot_id']}: długość klipu {measured_duration:.2f}s nie odpowiada "
                f"żądanemu ujęciu {float(shot['duration_sec']):.2f}s (tolerancja 0.20s)."
            )
        width = int(media['video'].get('width') or 0)
        height = int(media['video'].get('height') or 0)
        if width <= 0 or height <= 0 or abs((width / height) - (9 / 16)) > 0.025:
            raise RuntimeError(f"{shot['shot_id']}: klip nie ma poprawnego pionowego formatu 9:16 ({width}x{height}).")

        # Ustandaryzuj wynik do ścieżki z manifestu.
        if video_path.resolve() != output_path.resolve():
            output_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(video_path, output_path)
        record = {
            **result,
            'shot_id': shot['shot_id'],
            'scene_id': shot['scene_id'],
            'status': 'completed',
            'video_path': str(output_path),
            'duration_sec': measured_duration,
            'width': width,
            'height': height,
            'backend': result.get('backend', 'local'),
            'device': result.get('device', runtime_used['device']),
            'render_width': runtime_used['render_width'],
            'render_height': runtime_used['render_height'],
            'lipsync_applied': bool(result.get('lipsync_applied', False)),
            'checkpoint_path': str(checkpoint_path),
        }
        write_json_atomic(checkpoint_path, record)
        VISUAL_RECORDS.append(record)
        write_json_atomic(VISUAL_MANIFEST_PATH, {
            'schema_version': '1.0',
            'episode_id': EPISODE_ID,
            'completed_shots': len(VISUAL_RECORDS),
            'total_shots': len(SHOT_PLAN),
            'shots': VISUAL_RECORDS,
        })
        write_json_atomic(SHOT_PLAN_PATH, {
            'schema_version': '1.0',
            'episode_id': EPISODE_ID,
            'runtime': RENDER_RUNTIME,
            'shots': SHOT_PLAN,
        })
        print(f"[CHECKPOINT] Zapisano {shot['shot_id']} na Google Drive.")

    if len(VISUAL_RECORDS) != len(SHOT_PLAN):
        raise RuntimeError(
            f"Renderer zakończył {len(VISUAL_RECORDS)}/{len(SHOT_PLAN)} ujęć; "
            'gotowe checkpointy pozostają na Drive do wznowienia.'
        )
    print(f'[OK] Wszystkie ujęcia gotowe: {len(VISUAL_RECORDS)}')


In [ ]:
# KOMÓRKA 7/7 — montaż wszystkich ujęć, polskie napisy, FFmpeg i ffprobe

from IPython.display import display, Video


def srt_timestamp(seconds):
    milliseconds = max(0, int(round(seconds * 1000)))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, milliseconds = divmod(remainder, 1000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}'


def wrap_caption(text, width=30):
    return '\n'.join(textwrap.wrap(
        text.strip(), width=width,
        break_long_words=False, break_on_hyphens=False
    ))


if TEST_MODE:
    test_report = {
        'episode_id': EPISODE_ID,
        'test_mode': True,
        'mp4_created': False,
        'title': STORY['title'],
        'total_audio_seconds': round(TOTAL_DIALOGUE_SECONDS, 3),
        'goal_seconds': GOAL_VIDEO_SECONDS,
        'accepted_range_seconds': [MIN_VIDEO_SECONDS, MAX_VIDEO_SECONDS],
        'planned_shots': len(SHOT_PLAN),
        'shot_duration_range': [
            min(shot['duration_sec'] for shot in SHOT_PLAN),
            max(shot['duration_sec'] for shot in SHOT_PLAN)
        ],
        'shots': [
            {
                'shot_id': shot['shot_id'],
                'scene_id': shot['scene_id'],
                'duration_sec': shot['duration_sec'],
                'shot_kind': shot['shot_kind'],
                'video_output_path': shot['output_path'],
                'audio_clip_path': shot['audio_clip_path']
            }
            for shot in SHOT_PLAN
        ],
        'reason': 'TEST_MODE=True: scenario, Edge TTS and shot plan were prepared; local visual renderer and MP4 export were skipped.'
    }
    report_path = ASSETS_DIR / f'{EPISODE_ID}_test_report.json'
    write_json_atomic(report_path, test_report)
    print('\n[TEST_MODE] Scenariusz, audio, manifest i checkpointy planu zostały zapisane.')
    print(f'[INFO] Plan zawiera {len(SHOT_PLAN)} ujęć po {min(s["duration_sec"] for s in SHOT_PLAN):.2f}–{max(s["duration_sec"] for s in SHOT_PLAN):.2f}s.')
    print('[POMINIĘTO] Nie uruchomiono lokalnego modelu wizualnego i nie utworzono MP4.')
    print('[INFO] Raport:', report_path)

else:
    if len(VISUAL_RECORDS) != len(SHOT_PLAN):
        raise RuntimeError(
            f'Brakuje klipów: {len(VISUAL_RECORDS)}/{len(SHOT_PLAN)}. '
            'Uruchom ponownie komórkę 6 z RESUME_RENDERING=True.'
        )
    if len(TIMELINE) != len(SEGMENT_SPECS):
        raise RuntimeError('Brakuje audio dla części scen — przerwano eksport.')

    for item, timeline in zip(SEGMENT_SPECS, TIMELINE):
        audio_path = Path(timeline['audio_path'])
        if not audio_path.exists() or audio_path.stat().st_size < 4000:
            raise RuntimeError(f"{item['scene_id']}: brakuje pliku WAV przed montażem: {audio_path}.")
        audio_probe = probe_media(audio_path)
        if audio_probe['audio'] is None or audio_probe['video'] is not None:
            raise RuntimeError(f"{item['scene_id']}: plik dialogu nie jest prawidłowym audio WAV.")
        measured_audio = probe_duration(audio_path)
        if abs(measured_audio - float(timeline['duration'])) > 0.15:
            raise RuntimeError(
                f"{item['scene_id']}: WAV ma {measured_audio:.2f}s, "
                f"timeline zapisuje {timeline['duration']:.2f}s."
            )
    if not MIN_VIDEO_SECONDS <= TOTAL_DIALOGUE_SECONDS <= MAX_VIDEO_SECONDS:
        raise RuntimeError(
            f'Audio ma {TOTAL_DIALOGUE_SECONDS:.1f}s; wymagany zakres to {MIN_VIDEO_SECONDS}–{MAX_VIDEO_SECONDS}s.'
        )

    planned_visual_seconds = sum(float(shot['duration_sec']) for shot in SHOT_PLAN)
    if abs(planned_visual_seconds - TOTAL_DIALOGUE_SECONDS) > 0.15:
        raise RuntimeError(
            f'Plan obrazu ({planned_visual_seconds:.2f}s) nie zgadza się z audio ({TOTAL_DIALOGUE_SECONDS:.2f}s).'
        )

    normalized_video_paths = []
    for index, (shot, record) in enumerate(zip(SHOT_PLAN, VISUAL_RECORDS)):
        if record.get('shot_id') != shot['shot_id']:
            raise RuntimeError(
                f"{shot['shot_id']}: kolejność/identyfikator VISUAL_RECORDS nie zgadza się z manifestem."
            )
        source_path = Path(record.get('video_path') or shot['output_path'])
        if not source_path.exists() or source_path.stat().st_size < 10000:
            raise RuntimeError(f"{shot['shot_id']}: brak klipu źródłowego {source_path}.")
        source_probe = probe_media(source_path)
        if source_probe['video'] is None or source_probe['audio'] is not None:
            raise RuntimeError(
                f"{shot['shot_id']}: wejściowy klip musi zawierać obraz i nie może zawierać audio."
            )
        normalized_path = TEMP_DIR / f"{EPISODE_ID}_cut_{index:04d}_{shot['shot_id']}.mp4"
        target_duration = float(shot['duration_sec'])
        video_filter = (
            f'scale={VIDEO_WIDTH}:{VIDEO_HEIGHT}:force_original_aspect_ratio=increase,'
            f'crop={VIDEO_WIDTH}:{VIDEO_HEIGHT},fps={FPS},setsar=1,'
            'tpad=stop_mode=clone:stop_duration=3,'
            f'trim=duration={target_duration:.3f},setpts=PTS-STARTPTS,format=yuv420p'
        )
        run_cmd([
            'ffmpeg', '-y', '-i', str(source_path),
            '-vf', video_filter, '-an',
            '-t', f'{target_duration:.3f}',
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20',
            '-pix_fmt', 'yuv420p', '-r', str(FPS),
            str(normalized_path)
        ])
        if not normalized_path.exists() or normalized_path.stat().st_size < 10000:
            raise RuntimeError(f"{shot['shot_id']}: nie powstał znormalizowany klip.")
        normalized_video_paths.append(normalized_path)
        print(f"[OK] Ujęcie {index + 1}/{len(SHOT_PLAN)}: {shot['shot_id']} ({target_duration:.2f}s)")

    video_concat_list = TEMP_DIR / f'{EPISODE_ID}_video_cuts.txt'
    video_concat_list.write_text(
        '\n'.join(f"file '{path.as_posix()}'" for path in normalized_video_paths),
        encoding='utf-8'
    )
    joined_video = TEMP_DIR / f'{EPISODE_ID}_joined_video.mp4'
    run_cmd([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', str(video_concat_list),
        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '19',
        '-pix_fmt', 'yuv420p', '-r', str(FPS),
        '-an', str(joined_video)
    ])

    # Wszystkie WAV są już 48 kHz stereo PCM; concat zachowuje ciągłość dialogu.
    dialogue_probe = probe_media(dialogue_audio)
    if dialogue_probe['audio'] is None or dialogue_probe['video'] is not None:
        raise RuntimeError('Master dialogowy nie zawiera prawidłowego strumienia audio.')
    dialogue_duration = probe_duration(dialogue_audio)
    if abs(dialogue_duration - TOTAL_DIALOGUE_SECONDS) > 0.25:
        raise RuntimeError(
            f'Master audio ma {dialogue_duration:.2f}s, timeline {TOTAL_DIALOGUE_SECONDS:.2f}s.'
        )

    audio_concat_list = TEMP_DIR / f'{EPISODE_ID}_audio_lines.txt'
    audio_concat_list.write_text(
        '\n'.join(f"file '{Path(timeline['audio_path']).as_posix()}'" for timeline in TIMELINE),
        encoding='utf-8'
    )
    dialogue_audio = TEMP_DIR / f'{EPISODE_ID}_dialogue_master.wav'
    run_cmd([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', str(audio_concat_list),
        '-c:a', 'pcm_s16le', '-ar', '48000', '-ac', '2',
        str(dialogue_audio)
    ])

    subtitle_entries = []
    for item, timeline in zip(SEGMENT_SPECS, TIMELINE):
        speaker_label = SPEAKER_LABELS_PL.get(item['speaker'], 'GŁOS')
        caption = item['text'] if item['speaker'] == 'narrator' else f'{speaker_label}: {item["text"]}'
        subtitle_entries.append((
            float(timeline['start']),
            float(timeline['start']) + float(timeline['duration']),
            caption,
        ))

    srt_path = TEMP_DIR / f'{EPISODE_ID}_pl.srt'
    srt_lines = []
    for number, (start, end, caption) in enumerate(subtitle_entries, start=1):
        srt_lines.extend([
            str(number),
            f'{srt_timestamp(start)} --> {srt_timestamp(end)}',
            wrap_caption(caption, width=30),
            '',
        ])
    srt_path.write_text('\n'.join(srt_lines), encoding='utf-8')
    safe_srt_path = (
        srt_path.as_posix()
        .replace('\\', '\\\\')
        .replace(':', '\\:')
        .replace("'", "\\'")
    )
    subtitle_filter = (
        f"subtitles='{safe_srt_path}':"
        "force_style='FontName=DejaVu Sans,FontSize=44,PrimaryColour=&H00FFFFFF,"
        "OutlineColour=&HCC000000,BorderStyle=1,Outline=3,Shadow=1,Alignment=2,"
        "MarginV=340,MarginL=90,MarginR=90,WrapStyle=2'"
    )

    ambient_source = (
        f"anoisesrc=color=pink:amplitude=0.006:sample_rate=48000:"
        f"duration={TOTAL_DIALOGUE_SECONDS + 1.0:.3f}"
    )
    audio_mix = (
        '[0:a]aresample=48000,highpass=f=55,lowpass=f=16000,'
        'acompressor=threshold=0.10:ratio=2.0:attack=18:release=220[voice];'
        '[1:a]highpass=f=45,lowpass=f=620,'
        f'volume={AMBIENT_MIX_GAIN:.3f}[ambient];'
        "[voice][ambient]amix=inputs=2:duration=first:weights='1 0.18':normalize=0,"
        'alimiter=limit=0.95[aout]'
    )

    OUTPUT_VIDEO = OUTPUT_DIR / f'{EPISODE_ID}.mp4'
    run_cmd([
        'ffmpeg', '-y',
        '-i', str(joined_video),
        '-i', str(dialogue_audio),
        '-f', 'lavfi', '-i', ambient_source,
        '-vf', subtitle_filter,
        '-filter_complex', audio_mix,
        '-map', '0:v:0', '-map', '[aout]',
        '-t', f'{TOTAL_DIALOGUE_SECONDS:.3f}',
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '20',
        '-pix_fmt', 'yuv420p', '-r', str(FPS),
        '-c:a', 'aac', '-b:a', '192k', '-ar', '48000', '-ac', '2',
        '-movflags', '+faststart',
        str(OUTPUT_VIDEO)
    ])

    if not OUTPUT_VIDEO.exists() or OUTPUT_VIDEO.stat().st_size <= 100000:
        raise RuntimeError(f'Plik MP4 nie istnieje albo ma nieprawidłowy rozmiar: {OUTPUT_VIDEO}')
    probe = probe_media(OUTPUT_VIDEO)
    video_stream = probe['video']
    audio_stream = probe['audio']
    if not video_stream:
        raise RuntimeError('Walidacja ffprobe: brak strumienia wideo.')
    if not audio_stream:
        raise RuntimeError('Walidacja ffprobe: brak strumienia audio.')
    if int(video_stream.get('width', 0)) != VIDEO_WIDTH or int(video_stream.get('height', 0)) != VIDEO_HEIGHT:
        raise RuntimeError('Walidacja ffprobe: oczekiwano obrazu 1080×1920.')
    if video_stream.get('codec_name') != 'h264':
        raise RuntimeError('Walidacja ffprobe: oczekiwano H.264.')
    if video_stream.get('r_frame_rate') != f'{FPS}/1':
        raise RuntimeError(f"Walidacja ffprobe: oczekiwano {FPS} fps, wykryto {video_stream.get('r_frame_rate')}.")
    if audio_stream.get('codec_name') != 'aac':
        raise RuntimeError('Walidacja ffprobe: oczekiwano AAC.')
    if int(audio_stream.get('sample_rate', 0)) != 48000:
        raise RuntimeError(
            f"Walidacja ffprobe: oczekiwano audio 48 kHz, wykryto {audio_stream.get('sample_rate')}."
        )
    if int(audio_stream.get('channels', 0)) != 2:
        raise RuntimeError(
            f"Walidacja ffprobe: oczekiwano stereo, wykryto {audio_stream.get('channels')} kanałów."
        )
    if video_stream.get('pix_fmt') != 'yuv420p':
        raise RuntimeError(
            f"Walidacja ffprobe: oczekiwano yuv420p, wykryto {video_stream.get('pix_fmt')}."
        )

    video_duration = float(probe['data'].get('format', {}).get('duration', 0))
    if video_duration < MIN_VIDEO_SECONDS or video_duration > MAX_VIDEO_SECONDS:
        raise RuntimeError(
            f'Finalne MP4 ma {video_duration:.1f}s; dozwolony zakres to {MIN_VIDEO_SECONDS}–{MAX_VIDEO_SECONDS}s.'
        )
    if abs(video_duration - TOTAL_DIALOGUE_SECONDS) > 0.35:
        raise RuntimeError(
            f'Finalne MP4 ma {video_duration:.2f}s, ale ciągłe audio {TOTAL_DIALOGUE_SECONDS:.2f}s; '
            'sprawdź synchronizację montażu.'
        )
    if video_duration < GOAL_VIDEO_SECONDS:
        print(f'[OSTRZEŻENIE] Gotowy film ma mniej niż cel {GOAL_VIDEO_SECONDS}s.')

    loudness_check = run_cmd([
        'ffmpeg', '-i', str(OUTPUT_VIDEO), '-af', 'volumedetect', '-f', 'null', '-'
    ], check=False)
    loudness_text = (loudness_check.stderr or '') + (loudness_check.stdout or '')
    mean_match = re.search(r'mean_volume:\s*(-?inf|-?[0-9.]+)\s*dB', loudness_text)
    if not mean_match or mean_match.group(1) == '-inf':
        raise RuntimeError('Walidacja audio: wykryto cyfrową ciszę albo brak pomiaru.')
    print(f"[OK] Aktywny audio stream — mean_volume {mean_match.group(1)} dB")

    write_json_atomic(VISUAL_MANIFEST_PATH, {
        'schema_version': '1.0',
        'episode_id': EPISODE_ID,
        'total_shots': len(VISUAL_RECORDS),
        'total_duration_sec': video_duration,
        'shots': VISUAL_RECORDS,
    })
    print('\n' + '=' * 62)
    print('[GOTOWE] Lost-words / Zgubione słowa')
    print('Tytuł:', STORY['title'])
    print('Plik:', OUTPUT_VIDEO)
    print(f'Parametry: {VIDEO_WIDTH}×{VIDEO_HEIGHT}, {FPS} fps, H.264/AAC, {video_duration:.2f}s')
    print('Łączna liczba krótkich ujęć:', len(SHOT_PLAN))
    print('Sceny mówiące:', sum(1 for item in SEGMENT_SPECS if item['shot_type'] == 'talking_character'))
    print('Napisy: DejaVu Sans | Ambient: mix z dialogiem')
    print('=' * 62)
    display(Video(str(OUTPUT_VIDEO), embed=True, width=360))
